# Ablations

Produces Figure 13 of the paper (Appendix H.4, "Loss and low-rank comparison"):
LorD at ranks $r \in \{1, 2, 4, 8, 16\}$, fitted with the kernel score and with
the log score, measured against SCORE. The section states the question it
answers, reads what it needs from the results tree, and produces the figure.

1. [Low-rank expressivity against SCORE](#1-low-rank-expressivity-against-score)

## 1. Low-rank expressivity against SCORE

**The question.** LorD, the physical-space low-rank-plus-diagonal covariance
$\Sigma = D + UU^\top$ with $U \in \mathbb{R}^{d \times r}$, interpolates between
a diagonal model and an unstructured one. Two things should follow, and this
figure is built to show both:

1. **Expressivity grows with $r$.** As the rank rises the low-rank family should
   approach what an unstructured covariance can do — so its curve should fall
   towards SCORE rather than plateau early.
2. **The kernel score is the better objective for it.** At matched rank the
   KS-fitted LorD should beat the NLL-fitted one, and the gap should not be an
   artifact of a rank too small to matter.

**The baseline is SCORE.** $1\times$ *is* SCORE — $0$ nats on the log-score
panel, which is a difference rather than a multiple — and a curve above it is a
low-rank model that has not caught up. Which run that is depends on the task —
the graph-Fourier core with the Matérn filter on EUPPBench, the Fourier core
(stage-1 CRPS) everywhere else — so every entry in `TASKS` names its own
baseline instead of the choice being guessed from a slug prefix. It is drawn as
one horizontal with its own (data set, seed) spread as the band: the same
aggregation as the curves, so the band reads against the error bars rather than
being a second statistic drawn in the same panel.

### Design decisions

**Every run on the scale of its own task.** Scores differ by orders of
magnitude across tasks — the kernel score is $0.47$ on EUPPBench and $0.0019$ on
depth — so raw values cannot be pooled. Every run is therefore put on the scale
of the seed mean $\bar b_t$ of *its own* task's baseline, and `SCALE` decides
which scale that is:

* a **ratio** $v / \bar b_t$ for the five non-negative metrics. It is a
  unit-free multiple of SCORE's score — $1\times$ is SCORE, $1.3\times$ is thirty
  per cent worse — and it never changes sign, so a log axis can carry a panel
  whose arms are orders of magnitude apart.
* a **difference** $v - \bar b_t$, in nats per coordinate, for the log score,
  where $0$ is SCORE. That is scale-free for the same reason a ratio is: a
  difference of log densities is a log likelihood ratio, so $+0.7$ nats means a
  per-coordinate predictive density $e^{0.7} \approx 2\times$ smaller, whatever
  the task's own level is.

Both are oriented the same way — every metric here is lower-is-better, so above
SCORE is worse in both — which is what lets one figure carry the two. The ticks
of the five ratio panels carry a $\times$; the log-score panel states its unit
in its title instead.

**Three steps, and the last is robust.** Seeds are collapsed inside a task
first, so every task counts once whether it has five seeds or two. The tasks of
one data set — its modes and horizons, ETTm1's four or illness's two — are
collapsed next, by their median (`TASK_GROUPING`, `GROUP_STATISTIC`), so every
*data set* counts once however many variants of it were run. The point is then
the **median** of those data-set values and the bar their $0.25$–$0.75$
quantiles. Both depend on how many data sets disagree rather than on how far the
worst one strays: the NLL-fitted LorD can sit at $\approx 1\times$ on one data
set and $\approx 7\times$ on another, and a mean of such values is a number no
run produced. `aggregate(..., how="geometric")` and `"arithmetic"` keep the
mean-based versions.

Two cells have no data sets to take a quantile over and fall back to their
individual runs, which the `spread` column records: the **baseline**, whose
data-set means are all exactly the neutral value — $1$ on a ratio-scaled metric,
where it is its own denominator, $0$ on the log score — so the only spread it
has is seed-to-seed, and that is worth drawing, as the noise floor an arm has to
clear to be distinguishable from SCORE; and any panel resting on a **single
data set**.

**Read the box with the count.** A quartile over two data sets is an
interpolation between two numbers, so the box is narrower than the disagreement
it stands for. It becomes a quantile worth the name from about four data sets
on, which the pool clears: eleven data sets (twenty-seven tasks) at every
plotted rank, nine on the log score. Eleven is still few enough that one
dissenting data set moves the median, so `aggregate`'s `n_tasks` column is the
number to read the box against; `plot_rank_ablation_by_task` draws the
per-data-set view when a panel needs auditing.

**Why the log score is the one difference.** A ratio is orderable only against
a *positive* reference, and positive only if the run's own score carries the
same sign. The log score satisfies neither: it is negative on ERA5 and on most
univariate ETT tasks and positive on EUPPBench, illness, traffic and weather,
and depth's reference is positive while its low-rank arms score negative.
Dividing would leave that panel resting on EUPPBench alone. On the difference
scale the signs stop mattering and the panel keeps twenty-two of the
twenty-seven tasks — nine of the eleven data sets.

**What it still excludes: degenerate references.** SCORE's log score on NYU depth
is $+7$ to $+1.8 \times 10^{4}$ nats per coordinate across seeds, against $-1.6$
to $-3.3$ for every arm beside it — a near-singular covariance, not a bad
forecast. The results tables blank that same cell by the same rule
(`mrt.NLL_PER_D_LIMIT`, all-or-nothing per configuration, since seeds of one
configuration differ only by initialization), and `DEGENERATE` applies it here.
Keeping it would read as the low-rank model winning by thousands of nats, when
what it measures is the reference diverging. The same rule removes **ETTm2
multivariate** (both horizons; ETTm2 stays in the panel through its univariate
pair) and **Electricity** (all seeds of both horizons). All of them still
contribute to the other five panels, and the exclusion is printed with the rest.

**What it does not exclude: weather, where the same symptom stays under the
limit.** SCORE scores $+290$ to $+400$ nats per coordinate there, and so does
every KS-fitted Gaussian arm ($+230$ to $+920$), while the NLL-fitted arms score
$+0.5$ to $+7$. Every seed is under `NLL_PER_D_LIMIT`, so both horizons are
admitted. One data set of nine barely moves the median, but it is the most
extreme value in the panel, so the numbers without weather are given alongside.

**Both arms or neither.** A rank contributes a data set only when *both* swept
arms ran there. Otherwise the KS curve and the NLL curve would be averages over
different sets of data sets, and the gap between them — the comparison this
figure exists for — would be partly a difference of mixture rather than of
objective. `balanced_block` then keeps the largest (data set $\times$ rank)
block in which every admitted data set has every plotted rank; with complete
sweeps it keeps everything.

**All data sets, blanks included.** `TASKS` lists every data set of the paper.
A task whose runs are missing from the results tree is not an error — it
contributes nothing, and both `collect` and the per-metric report name it.

**Which data sets are plotted: eleven, as twenty-seven tasks.** Every task in
the pool has the complete rank ladder — `phys_lora` at $r = 1..16$ under both
losses and the rank-free SCORE arm to divide by — at five seeds, so the block
is $27 \times 5 = 135$ cells, which the median sees as eleven data sets: ETTh1,
ETTh2, ETTm1 and ETTm2 (each univariate and multivariate at
$T \in \{96, 192\}$), weather, electricity and traffic ($T \in \{96, 192\}$),
illness ($T \in \{24, 36\}$), EUPPBench, ERA5 and NYU depth. The log score rests
on nine data sets (twenty-two tasks), for the reasons above.

**Why the median is over data sets and not over tasks** (`TASK_GROUPING`):
little of the twenty-seven is independent — sixteen are ETT (two ETTh and two
ETTm data sets, each in two modes at two horizons), and two each are illness,
weather, electricity and traffic, at two horizons of one series. A median over
tasks was closer to eleven votes than to twenty-seven, and the ETT block alone
(sixteen of twenty-seven) could carry it; collapsing each data set's tasks
first makes the eleven votes explicit, ETT four of them.
`TASK_GROUPING = "dataset_mode"` keeps univariate and multivariate apart
(fifteen votes), and `None` restores one vote per task.
Depth is the data set that beats SCORE by the widest margin ($0.93\times$
on CRPS, $0.95$–$0.97\times$ on the kernel score); on the kernel score it is
joined below $1$ by illness ($0.97$–$0.99\times$ over its two horizons), traffic
($0.990$–$0.991\times$, both horizons at every rank), ETTm2
($0.993$–$0.999\times$; its multivariate horizons are below $1$ at every rank),
weather ($0.996$–$1.000\times$) and, marginally, EUPPBench ($0.997$–$0.999\times$)
— six of the eleven, which is why the KS curve's median sits just under the
SCORE line there ($0.998$–$1.000\times$, inside SCORE's own seed band at every
rank; electricity, at $1.01$–$1.02\times$, is with the five above). Per task the
same holds for fifteen of the twenty-seven, `etth2` multivariate, `etth1`
univariate at $T = 192$ and `ettm1` multivariate at $T = 96$ among them, but
inside data sets whose other tasks sit above $1$.

**Ranks.** Every task sweeps the ladder $r \in \{1,2,4,8,16\}$, so that is
`RANKS`.

**Six panels, one per metric,** in the column order of the results tables and
under their abbreviations (MSE, CRPS, ES, VS, KS, NLL — the same shorthand
`epoch_budget.ipynb` uses, because six panels abreast of the text width leave
about $1.1$ in each and "variogram score" overruns it). The x-axis and the
y-label are shared; the y-*unit* is not, which is why the log-score panel says
`[nats]` in its title and the five ratio panels put the $\times$ on their
ticks.

The y-*scales* stay per panel and turn **logarithmic** once a panel spans more
than $3\times$ (`LOG_ABOVE`), with ticks then labelled as multiples rather than
as powers of ten. The trigger is a *quantile*, not a maximum. On the current
pool four panels trigger it — CRPS, the energy score, the variogram score and
the kernel score — because the NLL-fitted arm has several far data sets (ERA5,
depth, electricity, traffic), where its low-rank fits are worse than a point
forecast; MSE stays linear. The outliers beyond the box are in
`plot_rank_ablation_by_task(df, metric)`.
The log-score panel is linear by construction: a difference in nats straddles
zero, so neither a log axis nor a multiplier applies to it.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _project_root(start=None):
    """Nearest ancestor with a `.project-root` marker -- same rule plot_config uses.

    Lets the notebook be run from the repository root or from `notebooks/`
    without the import below or the cache path depending on which.
    """
    start = Path.cwd() if start is None else Path(start)
    for parent in [start, *start.parents]:
        if (parent / ".project-root").exists():
            return parent
    return start


NB_DIR = _project_root() / "notebooks"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import plot_config as pc
import make_results_tables as mrt          # slug -> facet parsing, shared with the tables

CFG = pc.apply_style()

#: The results tree. This notebook reads *only* from here:
#: `<experiment>/<dataset>/[<mode>/][T<pred_len>/]<slug>/seed<N>/metrics.json`
#: is the record the paper's tables are built from, and a seed folder is
#: self-contained -- no W&B access, no local run directories, so the figure is
#: reproducible by anyone with read access to the folder.
from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

#: Every rank swept -- the `phys_lora` (LorD) ladder, r = 1..16, on every task.
#: A task contributes at the ranks it has, and the ladder actually drawn is
#: worked out per metric. No r=0 anchor: `phys_diag` (Diag) is a different
#: method, and this section sweeps one.
RANKS = (1, 2, 4, 8, 16)

#: The SCORE run on a task, i.e. the denominator of every ratio in this section.
#: It is a per-task choice because the paper's SCORE core is: the graph Fourier
#: transform with a Matern filter on EUPPBench, the Fourier core on everything
#: gridded or sequential. `stage1` is part of the identity --
#: `whitened_spectral_diag_kernel_score` and
#: `whitened_spectral_diag_s1crps_kernel_score` are two different runs.
_WHITENED_CRPS = dict(method="whitened_spectral_diag", stage1="crps",
                      loss="kernel_score")

#: Every dataset of the paper: a task with no readable `metrics.json` is not an
#: error, it simply contributes nothing and is named in the report. `path` is the
#: directory that *contains* the config-slug folders, given explicitly because
#: the experiments nest differently -- station postprocessing has no mode level,
#: and every time-series task sits at `<dataset>/<mode>/T<pred_len>/`.
TASKS = [
    dict(key="euppbench", d=115, label="EUPPBench",
         path=SHARED / "station_postprocessing" / "euppbench",
         baseline=dict(method="whitened_graph_matern", stage1="crps",
                       loss="kernel_score")),
    dict(key="era5", d=160 * 220, label="ERA5",
         path=SHARED / "surface_temperature" / "era5",
         baseline=_WHITENED_CRPS),
    # 426x560, the Eigen-cropped `nyu_bts_crop` event. Only `nll_per_d` reads it.
    dict(key="depth", d=426 * 560, label="NYU depth",
         path=SHARED / "depth" / "nyu_bts", baseline=_WHITENED_CRPS),
    # ETT (seeds 1-5, `data_seed` 42): both modes at the two reported horizons,
    # T96 and T192. Every one of these trains `phys_lora` at r = 1..16 under
    # *both* losses and the **rank-free**
    # `whitened_spectral_diag_s1crps_kernel_score`, the denominator every ratio
    # here needs.
    *[dict(key=f"{ds}_{f}_T{T}", d=T * (7 if f == "M" else 1),
           label=rf"{name} ({'univ.' if f == 'S' else 'multiv.'}) $T{{=}}{T}$",
           path=SHARED / "time_series" / ds / mode / f"T{T}",
           baseline=_WHITENED_CRPS, dataset=ds, mode=f)
      for ds, name in (("etth1", "ETTh1"), ("etth2", "ETTh2"),
                       ("ettm1", "ETTm1"), ("ettm2", "ETTm2"))
      for f, mode in (("S", "univariate"), ("M", "multivariate"))
      for T in (96, 192)],
    # Weather (seeds 1-5), multivariate, at the same horizon pair.
    *[dict(key=f"weather_T{T}", d=T * 21, label=rf"Weather $T{{=}}{T}$",
           path=SHARED / "time_series" / "weather" / "multivariate" / f"T{T}",
           baseline=_WHITENED_CRPS, dataset="weather", mode="M")
      for T in (96, 192)],
    # Electricity (seeds 1-5): 321 channels, multivariate only, at the same
    # horizon pair.
    *[dict(key=f"electricity_T{T}", d=T * 321, label=rf"Electricity $T{{=}}{T}$",
           path=SHARED / "time_series" / "electricity" / "multivariate" / f"T{T}",
           baseline=_WHITENED_CRPS, dataset="electricity", mode="M")
      for T in (96, 192)],
    # Traffic (seeds 1-5): 862 channels, multivariate only, at the same horizon
    # pair.
    *[dict(key=f"traffic_T{T}", d=T * 862, label=rf"Traffic $T{{=}}{T}$",
           path=SHARED / "time_series" / "traffic" / "multivariate" / f"T{T}",
           baseline=_WHITENED_CRPS, dataset="traffic", mode="M")
      for T in (96, 192)],
    # Illness (seeds 1-5, horizons T24 and T36), multivariate only, with the same
    # rank ladder under both losses and the rank-free SCORE denominator.
    dict(key="illness_T24", d=24 * 7, label=r"Illness $T{=}24$",
         path=SHARED / "time_series" / "national_illness" / "multivariate" / "T24",
         baseline=_WHITENED_CRPS, dataset="illness", mode="M"),
    dict(key="illness_T36", d=36 * 7, label=r"Illness $T{=}36$",
         path=SHARED / "time_series" / "national_illness" / "multivariate" / "T36",
         baseline=_WHITENED_CRPS, dataset="illness", mode="M"),
]

#: One panel per metric, in the column order of the paper's results tables
#: (`make_results_tables.COLUMNS`), named by the same abbreviations the results
#: tables and the runtime figures use (`epoch_budget.ipynb::SHORT_TITLE`): six
#: panels abreast of the text width leave about 1.1 in each, which "variogram
#: score" overruns. All are lower-is-better. `nll_per_d` is the log score
#: divided by the event dimension, the form the tables report.
METRICS = [
    ("mse", "MSE"),
    ("crps", "CRPS"),
    ("energy_score", "ES"),
    ("variogram_score", "VS"),
    ("kernel_score", "KS"),
    ("nll_per_d", r"NLL / $d$"),
]

#: How a metric is put on SCORE's scale -- the one thing that is not shared by
#: all six panels.
#:
#: Five of them are non-negative by construction, so a **ratio** to SCORE's
#: score is the natural unit: it is dimensionless, it pools across tasks whose
#: scores differ by orders of magnitude, and 1 is SCORE.
#:
#: The log score is not non-negative (about -1.3 nats per coordinate on ERA5,
#: +1.5 on EUPPBench), so a ratio to it is not even orderable: a negative
#: reference mirrors the axis, and a run whose score has the opposite sign to
#: the reference lands at a negative multiple. A **difference** in nats per
#: coordinate is the right unit there and is scale-free for the same reason a
#: ratio is -- it is a log likelihood ratio per coordinate, so 0.7 nats means
#: e^0.7 ~ 2x the per-coordinate predictive density, whatever the task's level.
#: Both are oriented the same way, up is worse: 1x and 0 are SCORE.
SCALE = {"nll_per_d": "difference"}

#: Unit of a difference-scaled metric, shown in its panel title -- the one place
#: the figure's axes do not all mean the same thing.
DIFF_UNIT = {"nll_per_d": "nats"}

#: |value| past which a configuration is degenerate rather than merely bad, per
#: metric. Only the log score has one, and it is the results tables' own limit
#: (`mrt.NLL_PER_D_LIMIT`), applied with the tables' all-or-nothing rule: seeds
#: of a configuration differ only by initialization, so one diverging seed means
#: the whole configuration's density is unstable. This keeps NYU depth out of the
#: NLL panel, where SCORE's log score diverges on some seeds and the differences
#: would read as LorD winning by thousands of nats. The tables blank that cell
#: for the same reason (see `mrt.aggregate`).
DEGENERATE = {"nll_per_d": mrt.NLL_PER_D_LIMIT}


def scale_of(metric):
    """`"ratio"` unless `SCALE` says the metric is compared by difference."""
    return SCALE.get(metric, "ratio")


#: What counts as **one vote** in the median over data sets. The tasks are not
#: independent: ETTm1 alone is four of them (two modes x two horizons). With
#: **`"dataset"`** every cell's per-task seed means are first collapsed within
#: each data set by `GROUP_STATISTIC` -- ETTm1 to the median of its four tasks,
#: illness and weather to that of their two horizons, EUPPBench, ERA5 and depth
#: unchanged -- so each data set counts once, whatever number of (mode, horizon)
#: variants it was run in. `"dataset_mode"` keeps univariate and multivariate
#: apart and collapses only the horizons; `None` is one vote per task. A task's
#: data set and mode are the `dataset` / `mode` of its `TASKS` entry, and a task
#: without them is a data set of its own.
TASK_GROUPING = "dataset"

#: How the tasks of one group are collapsed. A group has one, two or four
#: tasks, so its median is at most the mean of its middle two -- close to
#: "average over the horizons", without letting one odd task set its data set.
GROUP_STATISTIC = "median"

_TASK_BY_KEY = {t["key"]: t for t in TASKS}


def task_group(key, grouping="default"):
    """The group task `key` votes in under `grouping` (default `TASK_GROUPING`)."""
    grouping = TASK_GROUPING if grouping == "default" else grouping
    task = _TASK_BY_KEY.get(key, {})
    dataset, mode = task.get("dataset", key), task.get("mode")
    if grouping is None:
        return key
    if grouping == "dataset":
        return dataset
    if grouping == "dataset_mode":
        return dataset if mode is None else f"{dataset}_{mode}"
    raise ValueError(f"unknown TASK_GROUPING {grouping!r}")

#: The two swept arms. Colour carries the *fitting objective* throughout the
#: paper (plot_config.COLORS), so the same green means "kernel score" here and
#: in every other figure.
ARMS = [
    dict(method="phys_lora", loss="kernel_score", label="LorD, KS",
         color=pc.COLORS["kernel_score"], marker="o"),
    dict(method="phys_lora", loss="nll", label="LorD, NLL",
         color=pc.COLORS["log_score"], marker="s"),
]

#: How the baseline is drawn: one horizontal at the mean of its own ratios --
#: which is 1 by construction when every task has the same number of seeds --
#: with its (dataset, seed) spread as the band. Same style in every panel.
BASELINE = dict(label="SCORE", color=pc.COLORS["deterministic"],
                linestyle="-")

#: Default metric of the single-panel figure; every plotting function takes it
#: as an argument, so this is only the starting point.
METRIC = "kernel_score"

### 1.1 Reading the results tree

Everything comes from the results tree (`results_dir` in the path config) — no
W&B, no local run directories, so this figure is reproducible by anyone with
read access to that folder. Each `seed<N>/metrics.json` carries that run's final
test scores and is self-contained.

Two consequences worth knowing before reading the coverage table below:

* A configuration is invisible here until it is **synced**: the notebook reads
  `metrics.json`, which `tools/sync_checkpoints.py` writes next to each
  checkpoint.
* A missing cell is a **gap**, not an error. Absent ranks leave gaps in the
  curves and absent data sets drop out of the pool, both reported rather than
  silently absorbed.

The two nestings of the tree (`<dataset>/` and `<dataset>/<mode>/T<pred_len>/`)
are handled by giving every task its `path` explicitly, so the reader never has
to guess a layout from a dataset name.

In [ ]:
#: Test scores carried by every `metrics.json`. Non-finite entries (a Dirac's
#: log score) are stored as null there and read back as NaN.
_RAW_METRICS = ("nll", "crps", "energy_score", "kernel_score",
                "variogram_score", "mse", "mae")

_ROW_COLUMNS = ["task", "d", "slug", "method", "loss", "stage1", "rank",
                "seed", *_RAW_METRICS]


def read_task(task):
    """One row per synced seed of one task, read from its `metrics.json` files.

    The facets (method, rank, stage-1 loss, objective) come from the config-slug
    directory name via the same parser the paper's tables use, so the slug
    convention is defined in exactly one place. A seed folder without
    `metrics.json` is skipped and counted -- checkpoints synced before
    2026-08-14 have no such file, and those runs are invisible here until the
    sync is re-run for them.
    """
    rows, pending = [], 0
    if not task["path"].is_dir():
        return rows, pending
    for slug_dir in sorted(task["path"].iterdir()):
        if not slug_dir.is_dir():
            continue
        cfg = mrt.Config(mrt._slug_to_tags(slug_dir.name))
        for seed_dir in sorted(slug_dir.glob("seed*")):
            mfile = seed_dir / "metrics.json"
            if not mfile.is_file():
                pending += 1
                continue
            test = json.loads(mfile.read_text()).get("test", {})
            row = dict(task=task["key"], d=task["d"], slug=slug_dir.name,
                       method=cfg.method, loss=cfg.objective(),
                       stage1=cfg.stage1,
                       rank=int(cfg.rank) if cfg.rank else None,
                       seed=seed_dir.name[4:])
            row.update({m: mrt._num(test.get(m)) for m in _RAW_METRICS})
            rows.append(row)
    return rows, pending


def collect(tasks=TASKS, verbose=True):
    """Every synced run of the tasks above, one row per (configuration, seed).

    Not cached on purpose: this is a walk over a few hundred small JSON files,
    so it is cheaper to re-read than to reason about whether a cache predates
    the last `sync_checkpoints.py` run.
    """
    rows = []
    for task in tasks:
        got, pending = read_task(task)
        rows += got
        if verbose:
            state = "no such directory" if not task["path"].is_dir() else \
                    f"{len(got)} runs, {pending} seed dirs without metrics.json"
            print(f"[{task['key']}] {state}")
    df = pd.DataFrame(rows, columns=_ROW_COLUMNS)
    # The log score per coordinate, the form the paper's tables report.
    df["nll_per_d"] = df["nll"] / df["d"]
    return df


def _cell(df, task_key, spec, rank, metric):
    """The seeds and values of one (task, configuration) cell.

    ``spec`` is an arm or a task's baseline: method, objective and -- when the
    spec names one -- the stage-1 loss, which is what separates the two
    `whitened_spectral_diag` kernel-score runs from each other.
    """
    sel = df[(df["task"] == task_key) & (df["method"] == spec["method"])
             & (df["loss"] == spec["loss"])]
    if spec.get("stage1") is not None:
        sel = sel[sel["stage1"] == spec["stage1"]]
    sel = sel[sel["rank"].isna()] if rank is None else sel[sel["rank"] == rank]
    sel = sel[np.isfinite(sel[metric].to_numpy(dtype=float))]
    return sel[["seed", metric]].rename(columns={metric: "value"})


def coverage(df, metric=METRIC, tasks=TASKS):
    """Seed counts of every cell this section needs -- 0 is a gap in the plot.

    Printed rather than hidden: the campaign is still filling in, and a curve
    with a silently absent rank looks the same as one that was never planned.
    Tasks with nothing at all are dropped from the columns to keep it readable;
    `collect` has already named them.
    """
    rows = [{"arm": a["label"], "rank": r,
             **{t["key"]: len(_cell(df, t["key"], a, r, metric)) for t in tasks}}
            for a in ARMS for r in RANKS]
    rows.append({"arm": BASELINE["label"], "rank": "--",
                 **{t["key"]: len(_cell(df, t["key"], t["baseline"], None, metric))
                    for t in tasks}})
    out = pd.DataFrame(rows).set_index(["arm", "rank"])
    return out.loc[:, out.any(axis=0)]

The coverage table is the honest status of this section: every cell the figure
needs, and how many seeds back it. A `0` becomes a gap in a curve or a dropped
data set — never a quietly interpolated point. Columns with nothing at all are
left out; `collect` above has already named those tasks.

In [ ]:
df = collect()
coverage(df)

### 1.2 Normalization and aggregation

For task $t$, arm $a$, rank $r$ and seed $s$, with $v_{t,s}(a,r)$ the metric
of that run and $\bar b_t$ the seed mean of $t$'s own SCORE baseline:

$$
\rho_{t,s}(a,r) =
\begin{cases}
  v_{t,s}(a,r) \,/\, \bar b_t & \text{ratio scale (MSE, CRPS, ES, VS, KS),}\\[2pt]
  v_{t,s}(a,r) - \bar b_t     & \text{difference scale (NLL, in nats per coordinate),}
\end{cases}
\qquad
\bar\rho_t(a,r) = \operatorname*{mean}_s \rho_{t,s}(a,r),
$$

and, with $g(t)$ the data set task $t$ belongs to (`TASK_GROUPING`: ETTm1
univariate and multivariate at $T = 96$ and $192$ are one data set, illness at
$T = 24$ and $36$ another),

$$
\bar\rho_g(a,r) = \operatorname*{median}_{t \in T(r),\ g(t) = g} \bar\rho_t(a,r),
\qquad
m(a,r) = \operatorname*{median}_{g \in G(r)} \bar\rho_g(a,r),
\qquad
\text{bar}(a,r) = \Big[\, q_{0.25},\ q_{0.75} \,\Big]
\ \text{of}\ \big\{ \bar\rho_g(a,r) \big\}_{g \in G(r)} .
$$

SCORE is $\rho \equiv 1$ on the first scale and $\rho \equiv 0$ on the second,
and on both, above SCORE is worse. Only the two lines above differ between the
scales; everything after them — the seed mean, the median within a data set,
the median over data sets, the quartile bar — is the same arithmetic, which is
why the six panels can be read against one another at all.

$T(r)$ is the set of tasks admitted at rank $r$: those of the balanced block
that pass this metric's admission rule and have both arms at $r$; $G(r)$ is the
set of data sets they belong to. The rule is the scale's own — a positive
reference **and** positive scores for a ratio, a non-degenerate one
($|v| \le$ `DEGENERATE[metric]`) for a difference — and it is applied per task,
so a data set can enter a panel through some of its tasks only (ETTm2 on the log
score, through its univariate pair). For the baseline, and for a panel with a
single admitted data set, the median and the quantiles are taken over the
individual $\rho_{t,s}$ instead — `aggregate` records which in its `spread`
column, so a bar is never read as a disagreement between data sets when it is
the seed noise of one.

In [ ]:
def rank_coverage(df, metric, tasks=TASKS, verbose=True):
    """`{task: (reference, ranks)}` for every task this metric can use.

    The reference is the seed mean of that task's standardized arm. A task drops
    out here for one of three reasons, all reported: no baseline run, a
    reference this metric's scale cannot be taken against, or no rank at which
    *both* swept arms exist.

    What the second one means depends on `scale_of(metric)`, and the difference
    between the two is why the log-score panel is not a ratio:

    * a **ratio** needs a strictly positive reference *and* strictly positive
      values. Dividing by a negative denominator would silently flip the
      direction of the axis, turning "approaches the baseline from above" into
      its mirror image; and a run whose score has the opposite sign to the
      reference has a *negative* ratio, which is neither orderable against the
      others nor drawable on the log axis the wide panels need. The log score is
      the only metric that can do either -- it is about -1.3 nats per coordinate
      on ERA5 and +1.5 on EUPPBench, and depth's low-rank arms score negative
      against a positive reference -- so on the ratio-scaled panels this guard
      never fires.
    * a **difference** needs neither, which is the whole point of it: it only
      needs finite values that are not degenerate. `DEGENERATE` sets that limit
      (the results tables' own `NLL_PER_D_LIMIT`, applied all-or-nothing per
      task), and it is what keeps NYU depth out of the log-score panel.

    Requiring both arms at a rank is what keeps the two curves comparable: a
    rank where only the KS arm ran would put that task into one curve and not
    the other, and the gap between them would then be partly a difference of
    task mixture rather than of objective.
    """
    out, notes = {}, {}
    scale, limit = scale_of(metric), DEGENERATE.get(metric)
    for t in tasks:
        base = _cell(df, t["key"], t["baseline"], None, metric)
        if base.empty:
            notes[t["key"]] = "no baseline run"
            continue
        ref = float(base["value"].mean())
        if not np.isfinite(ref):
            notes[t["key"]] = f"baseline {metric} is not finite"
            continue
        # Every value the task contributes, both arms and the baseline, at every
        # rank: the guards are about whether the task's *cells* can be put on one
        # scale, and a single run is enough to break either of them.
        seen = [v for a in [t["baseline"], *ARMS]
                for r in (None, *RANKS)
                for v in _cell(df, t["key"], a, r, metric)["value"]]
        if scale == "ratio":
            if ref <= 0:
                notes[t["key"]] = (f"baseline {metric} = {ref:.4g} is not positive, "
                                   "so a ratio to it would invert the axis")
                continue
            if any(v <= 0 for v in seen):
                notes[t["key"]] = (f"some {metric} values are not positive, so their "
                                   "ratio to the baseline would be negative")
                continue
        elif limit is not None and seen and max(abs(v) for v in seen) > limit:
            worst = max(seen, key=abs)
            notes[t["key"]] = (f"some |{metric}| exceed {limit:g} (worst {worst:.4g}): "
                               "a degenerate density rather than a bad one, so the "
                               "difference would measure the divergence")
            continue
        ranks = {r for r in RANKS
                 if all(not _cell(df, t["key"], a, r, metric).empty for a in ARMS)}
        if not ranks:
            notes[t["key"]] = "no rank with both LorD arms"
            continue
        out[t["key"]] = (ref, ranks)
    if verbose:
        grouped = {}
        for key, why in notes.items():
            grouped.setdefault(why, []).append(key)
        for why, keys in grouped.items():
            print(f"    excluded {', '.join(keys)}: {why}")
    return out


def balanced_block(cover):
    """The largest (tasks x ranks) block in which every task has every rank.

    Without it the averaged task set changes from rank to rank and a step in
    the curve is partly coverage rather than expressivity -- with ERA5 having no
    r=8 run, the NLL arm's ratio would fall from 3.9 to 1.0 at r=8 and climb
    back at r=16 purely because the one task whose NLL fit degenerates is
    missing in the middle of the ladder.

    Tasks are tried widest-ladder-first and the block that covers the most
    (task, rank) cells wins, so the rule costs nothing once the campaign is
    complete -- then every task has every rank and the block is everything.
    """
    order = sorted(cover, key=lambda k: (-len(cover[k][1]), k))
    keys, ranks = [], set()
    for k in range(1, len(order) + 1):
        candidate = order[:k]
        common = set.intersection(*[cover[x][1] for x in candidate])
        if len(candidate) * len(common) > len(keys) * len(ranks):
            keys, ranks = candidate, common
    return keys, ranks


def ratios(df, metric, tasks=TASKS, balanced=True, verbose=True):
    """One row per (cell, dataset, seed): its score against that dataset's baseline.

    This is the unit of the figure. Scores differ by orders of magnitude across
    tasks -- the kernel score is 0.47 on EUPPBench and 0.0019 on depth -- so raw
    values cannot be pooled; putting every run on the scale of its *own* task's
    standardized arm is unit-free and pools cleanly. It happens per run, not per
    task mean, so the seed spread rides along on the same unit-free scale.

    The `rel` column is that quantity, and which of the two it is comes from
    `scale_of(metric)`: the **ratio** `v / b` for the five non-negative metrics,
    where 1 is our method and 1.3 is thirty per cent worse, and the
    **difference** `v - b` in nats per coordinate for the log score, where 0 is
    our method and +0.3 is a per-coordinate predictive density e^0.3 ~ 1.35x
    smaller. Both are oriented the same way -- every metric here is
    lower-is-better, so above ours is worse in both -- which is what lets one
    figure carry both.

    `improvement` is the positively oriented form of the same number, kept for
    reading off a cell rather than for plotting: `100 (1 - ratio)` per cent on a
    ratio-scaled metric, `b - v` nats on a difference-scaled one.
    """
    scale = scale_of(metric)
    cover = rank_coverage(df, metric, tasks, verbose)
    columns = ["arm", "kind", "rank", "task", "seed", "rel", "improvement"]
    if not cover:
        return pd.DataFrame(columns=columns)
    keys, ladder = balanced_block(cover) if balanced else \
        (list(cover), set().union(*[v[1] for v in cover.values()]))
    if verbose:
        dropped = sorted(set(cover) - set(keys))
        if dropped:
            print(f"    dropped {', '.join(dropped)}: no common rank ladder "
                  f"with {', '.join(sorted(keys))}")
        print(f"    {len(keys)} data set{'s' if len(keys) > 1 else ''} "
              f"({', '.join(sorted(keys))}) on ranks {sorted(ladder)}")
    by_key = {t["key"]: t for t in tasks}
    rows = []
    for key in keys:
        ref, own = cover[key]
        cells = [(BASELINE["label"], "baseline", np.nan, by_key[key]["baseline"])]
        cells += [(a["label"], "arm", float(r), a)
                  for r in sorted(ladder & own) for a in ARMS]
        for label, kind, rank, spec in cells:
            got = _cell(df, key, spec,
                        None if kind == "baseline" else int(rank), metric)
            for _, row in got.iterrows():
                rel = (row["value"] / ref if scale == "ratio"
                       else row["value"] - ref)
                rows.append(dict(arm=label, kind=kind, rank=rank, task=key,
                                 seed=row["seed"], rel=rel))
    out = pd.DataFrame(rows, columns=["arm", "kind", "rank", "task", "seed",
                                      "rel"])
    out["improvement"] = (100.0 * (1.0 - out["rel"]) if scale == "ratio"
                          else -out["rel"])
    return out


def _task_means(rat):
    """Each data set's seed mean, one row per (cell, data set).

    The first of the two aggregation steps. Collapsing seeds inside a data set
    before combining data sets is what makes the second step a statement about
    *data sets*: every data set then counts once, whether it has five seeds or
    two, and the median below is over as many numbers as there are data sets.
    """
    return (rat.groupby(["arm", "kind", "rank", "task"], dropna=False)["rel"]
               .mean().reset_index())


def _group_center(values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if not v.size:
        return np.nan
    return float(np.median(v)) if GROUP_STATISTIC == "median" else float(v.mean())


def _group_means(per, grouping="default"):
    """The middle step: the tasks of one data set collapsed to one value.

    `per` is `_task_means`'s frame; the tasks sharing a `task_group` are
    collapsed with `GROUP_STATISTIC`, so ETTm1 (four tasks) and EUPPBench (one)
    each put one number into the median over data sets. From here on `task`
    holds the group and `n_variants` says how many tasks it stands for.
    `grouping=None` changes nothing but the column.
    """
    grouping = TASK_GROUPING if grouping == "default" else grouping
    out = per.assign(task=per["task"].map(lambda k: task_group(k, grouping)),
                     n_variants=1)
    if grouping is None:
        return out
    return (out.groupby(["arm", "kind", "rank", "task"], dropna=False)
               .agg(rel=("rel", _group_center), n_variants=("n_variants", "sum"))
               .reset_index())


def aggregate(rat, how="median", scale="ratio"):
    """Centre and spread of every cell: three steps, data sets last.

    Seeds are averaged within a task (`_task_means`), the tasks of one data set
    -- its modes and horizons -- are collapsed to one value (`_group_means`,
    `TASK_GROUPING`), and only then is the median taken, over data sets.

    The ratios are not one cloud -- on the kernel score at r=16 EUPPBench sits
    at 1.01x and ERA5 at 6.9x -- so the statistic has to survive one data set
    behaving completely differently from the others. The default is therefore
    the **median** of the data-set means, with the quartiles as the bar: both
    are unaffected by how extreme an outlying data set is, only by how many
    there are, which is the robustness a mean and sd did not have here (an
    arithmetic mean of those two numbers is 3.97, a value no run has).

    Two cells cannot take the quartiles over data sets, and both fall back to
    their individual runs:

    * the **baseline**, whose data-set means are all exactly the neutral value
      -- 1 on a ratio-scaled metric, where it is its own denominator, 0 on a
      difference-scaled one -- so the only spread it has is seed-to-seed, which
      is worth showing: it is the noise floor an arm has to beat to be
      distinguishable from ours;
    * any panel resting on a **single data set**, where there is nothing to take
      a quartile over. The `spread` column records which of the two a row used,
      so a bar is never read as a disagreement between data sets when it is the
      seed noise of one.

    `how="geometric"` and `"arithmetic"` keep the mean-based versions (the
    second is the plain mean and sd of the data-set means). `scale` is only
    there to refuse the first one on a difference-scaled metric, where a
    geometric mean of numbers straddling zero is not defined.
    """
    per = _group_means(_task_means(rat))
    # The baseline's rank is NaN, which never equals itself, so it cannot be a
    # dict key: the rank-free cells are filed under -1 instead.
    def _key(arm, kind, rank):
        return (arm, kind, float(rank) if np.isfinite(rank) else -1.0)

    if how == "geometric" and scale != "ratio":
        raise ValueError("a geometric mean needs a ratio; this metric is "
                         f"compared by {scale}")
    runs = {_key(*k): g.to_numpy(dtype=float) for k, g in
            rat.groupby(["arm", "kind", "rank"], dropna=False)["rel"]}
    rows = []
    for (arm, kind, rank), g in per.groupby(["arm", "kind", "rank"],
                                            dropna=False):
        v = g["rel"].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        over = "data sets"
        if kind == "baseline" or len(v) < 2:
            v = runs[_key(arm, kind, rank)]
            v, over = v[np.isfinite(v)], "runs"
        if how == "median":
            centre = float(np.median(v))
            lo, hi = (float(np.quantile(v, 0.25)), float(np.quantile(v, 0.75))) \
                if len(v) > 1 else (np.nan, np.nan)
            spread = np.nan
        elif how == "geometric":
            log = np.log(v)
            centre = float(np.exp(log.mean()))
            spread = float(np.exp(log.std(ddof=1))) if len(v) > 1 else np.nan
            lo, hi = ((centre / spread, centre * spread)
                      if np.isfinite(spread) else (np.nan, np.nan))
        else:
            centre = float(v.mean())
            spread = float(v.std(ddof=1)) if len(v) > 1 else np.nan
            lo, hi = ((centre - spread, centre + spread)
                      if np.isfinite(spread) else (np.nan, np.nan))
        rows.append(dict(arm=arm, kind=kind, rank=rank, mean=centre, sd=spread,
                         lo=lo, hi=hi, spread=over, n=int(len(v)),
                         n_runs=int(len(runs[_key(arm, kind, rank)])),
                         n_tasks=int(g["task"].nunique()),
                         n_variants=int(g["n_variants"].sum())))
    return pd.DataFrame(rows).sort_values(["kind", "arm", "rank"],
                                          ignore_index=True)


def _series(agg, label, ranks):
    """(mean, lo, hi) of one arm at every rank, NaN where the cell has no run."""
    sub = agg[agg["arm"] == label].set_index("rank")
    get = lambda col: np.array([sub[col].get(float(r), np.nan) for r in ranks])
    return get("mean"), get("lo"), get("hi")

### 1.3 The figure

`plot_metric_grid` puts both arms into one panel per metric, six panels in a
$2 \times 3$ grid; it is off by default (`DRAW_METRIC_GRID`
brings it back). `plot_arm_grid` draws the same panels with one row per arm, and
is the figure written (Figure 13 of the paper), twice: with the median and quartiles over data sets, and
with their mean and standard deviation. Every function takes the
metric list and the data set list as arguments, so restricting either is a
one-word edit, and each returns its figure, so any of them can be saved with
`pc.save(fig, "name.pdf")`.

In [ ]:
_ARM_STYLE = {a["label"]: a for a in ARMS}

#: Shared by every panel of the grid, and by the standalone panels. Deliberately
#: says "against" and not "relative to": five panels are a multiple of SCORE's
#: score ("1x is SCORE, 2x is twice the score", and their ticks carry the `x`)
#: and the log-score panel is a difference in nats per coordinate ("0 is SCORE").
#: Both are oriented the same way -- up is worse, as it is for every metric here
#: -- and the panel that is not a multiple says so in its title.
YLABEL = r"score against SCORE"


def metric_label(metric, unit=True):
    """Panel title: the metric's abbreviation, with its unit where it has one.

    Only a difference-scaled panel has a unit to state -- a ratio is a pure
    number -- so this is also what marks the one panel whose y-axis does not
    mean the same thing as its neighbours'.
    """
    label = dict(METRICS).get(metric, metric)
    unit_of = DIFF_UNIT.get(metric) if scale_of(metric) != "ratio" else None
    return f"{label} [{unit_of}]" if unit and unit_of else label


def _ladder(agg):
    """The ranks a panel actually drew, in order."""
    return [int(r) for r in sorted(agg[agg["kind"] == "arm"]["rank"].dropna().unique())]


def _draw(ax, agg, ranks, band=True):
    """The baseline as a horizontal with its spread, the two arms as curves."""
    base = agg[agg["kind"] == "baseline"]
    if not base.empty:
        m = float(base["mean"].iloc[0])
        lo, hi = float(base["lo"].iloc[0]), float(base["hi"].iloc[0])
        ax.axhline(m, color=BASELINE["color"], linestyle=BASELINE["linestyle"],
                   lw=CFG.thin_line_width, alpha=0.9, zorder=1,
                   label=BASELINE["label"])
        if band and np.isfinite(lo) and np.isfinite(hi):
            ax.axhspan(lo, hi, color=BASELINE["color"], alpha=0.12, lw=0,
                       zorder=0)
    for label, style in _ARM_STYLE.items():
        mean, lo, hi = _series(agg, label, ranks)
        if not np.isfinite(mean).any():
            continue
        yerr = np.vstack([mean - lo, hi - mean]) if band else None
        ax.errorbar(list(ranks), mean, yerr=yerr,
                    color=style["color"], marker=style["marker"], ms=3.0,
                    lw=CFG.line_width, capsize=2,
                    elinewidth=CFG.thin_line_width * 0.8, label=label, zorder=3)


def _decorate_x(ax, ranks):
    ax.set_xscale("log", base=2)
    ax.set_xticks(list(ranks))
    ax.set_xticklabels([str(r) for r in ranks])
    ax.minorticks_off()


#: Ratio span beyond which a panel goes logarithmic, and the tick ladders it
#: then uses: a 1-2-5 ladder while the panel still fits in one order of
#: magnitude or two, decades once it does not.
LOG_ABOVE = 3.0
_TICKS_NARROW = (0.2, 0.5, 1, 2, 5, 10, 20, 50)
_TICKS_WIDE = (0.01, 0.1, 1, 10, 100, 1000, 10000)


def _yscale(ax, agg, scale="ratio"):
    """Linear multiples of our score, unless the panel spans too many of them.

    The KS-fitted arm lands within a few per cent of ours while the NLL-fitted
    one is ~7x its score in the kernel score and ~200x in the variogram score.
    A linear axis wide enough for the second draws the first as a line on 1 with
    invisible error bars, so a panel whose values span more than `LOG_ABOVE`
    switches to a log axis -- on which a ratio is the natural quantity anyway:
    "twice the score" and "half the score" are then the same distance from ours.

    Every ratio is positive (`rank_coverage` admits only data sets whose scores
    and reference share a sign), so no symlog and no clipped whisker: the log
    axis is defined on the whole panel, bars included.

    Ticks are set explicitly and labelled as multiples, because the default log
    locator labels the wide panels `10^0, 10^1, 10^2` -- which reads as an
    exponent rather than "ten times the score" -- and gives the narrow ones a
    single decade tick with nothing in between. The linear branch appends the
    same `x` for the same reason it is on the log ticks: with one panel of the
    grid carrying nats and the rest multiples, a bare `1.02` is ambiguous.

    A **difference**-scaled panel is left alone: it is in nats per coordinate,
    it straddles zero, and neither a log axis nor a multiplier suffix applies.
    """
    if scale != "ratio":
        return
    v = np.concatenate([agg[c].to_numpy(dtype=float) for c in ("mean", "lo", "hi")])
    v = v[np.isfinite(v) & (v > 0)]
    if not len(v) or v.max() / v.min() <= LOG_ABOVE:
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda t, _: f"{t:g}" + r"$\times$"))
        return                                    # linear, autoscaled
    ax.set_yscale("log")
    ladder = _TICKS_WIDE if v.max() / v.min() > 30 else _TICKS_NARROW
    ticks = [t for t in ladder if v.min() / 1.5 <= t <= v.max() * 1.5]
    if ticks:
        ax.set_yticks(ticks)
        ax.set_yticklabels([f"{t:g}" + r"$\times$" for t in ticks])
    ax.set_yticks([], minor=True)


def _panel(ax, df, metric, tasks=TASKS, balanced=True, verbose=True):
    """One metric into one axes. Returns `(agg, ranks)`, or `(None, None)`."""
    rat = ratios(df, metric, tasks, balanced, verbose)
    if rat.empty:
        ax.text(0.5, 0.5, "no runs yet", transform=ax.transAxes, ha="center",
                va="center", color=pc.COLORS["guide"],
                fontsize=CFG.annotation_size)
        ax.set_yticks([])
        return None, None
    scale = scale_of(metric)
    agg = aggregate(rat, scale=scale)
    ranks = _ladder(agg)
    _draw(ax, agg, ranks)
    _yscale(ax, agg, scale)
    return agg, ranks


def _keys(axes):
    """The figure's legend entries, in the order the panels introduce them."""
    handles, labels = [], []
    for ax in np.atleast_1d(axes).ravel():
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                handles.append(h)
                labels.append(l)
    return handles, labels


def _footer(fig, axes, xlabel, ylabel=YLABEL, ncol=None, pad=0.012):
    """Axis label and a boxed key in a strip carved out under the panels.

    `loc="outside lower center"` is the one legend position constrained layout
    reserves space for, but it hands the legend and `fig.supxlabel` the *same*
    strip and draws them on top of each other, and `rect` does not help --
    the engine places `supxlabel` in the figure's bottom margin, outside the
    rect. So both are placed by hand, the way `epoch_budget.ipynb::plot_ablation`
    does it: the key pinned to the bottom of the canvas, the label just above
    it, and the engine told to keep the strip they need clear. Nothing leaves
    the canvas, so `pc.save` keeps its no-tight-bbox rule -- a tight bounding
    box would silently change the paper width.

    The strip is *measured* on a first layout pass rather than guessed: how much
    room the two need follows the font size, the number of entries and the
    figure height, all of which move. `fig.text` is not managed by the layout
    engine, so the label stays where it is put across later draws and `savefig`.

    `fig.supylabel` is then placed by hand too, for a reason the y-label alone
    would not show: it defaults to the middle of the *canvas*, and the canvas
    now has a footer, so on a two-row figure it would sit visibly below the
    panels it names.
    """
    handles, labels = _keys(axes)
    label = fig.text(0.5, 0.0, xlabel, ha="center", va="bottom",
                     fontsize=CFG.label_size)
    legend = fig.legend(handles, labels, loc="lower center",
                        bbox_to_anchor=(0.5, 0.0), bbox_transform=fig.transFigure,
                        ncol=ncol or len(handles), fontsize=CFG.legend_size,
                        frameon=True, fancybox=True, framealpha=1.0,
                        borderpad=0.6, columnspacing=1.6, handlelength=1.6,
                        edgecolor="0.7")
    fig.draw_without_rendering()
    to_fig = fig.transFigure.inverted()
    legend_box = legend.get_window_extent().transformed(to_fig)
    label_box = label.get_window_extent().transformed(to_fig)
    label.set_position((0.5, legend_box.y1 + pad))
    strip = legend_box.y1 + label_box.height + 2 * pad
    fig.get_layout_engine().set(rect=(0, strip, 1, 1 - strip))
    if ylabel:
        fig.supylabel(ylabel, fontsize=CFG.label_size, y=strip + (1 - strip) / 2)
    return legend, label


def plot_metric_grid(df, metrics=METRICS, tasks=TASKS, balanced=True, ncols=3,
                     aspect=0.55, save_as=None, verbose=True):
    """The headline figure: one panel per metric, every dataset pooled.

    Every panel puts the arms on the scale of the same baseline, oriented the
    same way: ours is the horizontal, above it is worse. The x-axis is shared
    and so is the y-label -- five panels are a multiple of our score and the
    log-score panel a difference in nats per coordinate, which the axis name
    covers and the panel title spells out. The y-*scales* stay per panel: the
    metrics differ by orders of magnitude in how far the degenerate arm strays
    (1.2x in MSE, 100x in the variogram score), and one shared scale would show
    a single panel and flatten five.

    Returns the figure, the axes and the aggregate frame of each metric, so the
    numbers behind any point can be read off without redrawing.
    """
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = pc.subplots(nrows, ncols, aspect=aspect, columns="double",
                            sharex=True, squeeze=False)
    aggs, ladders = {}, []
    for ax, (metric, _label) in zip(axes.flat, metrics):
        if verbose:
            print(f"[{metric}]")
        ax.set_title(metric_label(metric))
        agg, ranks = _panel(ax, df, metric, tasks, balanced, verbose)
        if agg is None:
            continue
        aggs[metric] = agg
        ladders += ranks
    for ax in axes.flat:
        _decorate_x(ax, sorted(set(ladders)))
    for ax in axes.flat[len(metrics):]:
        ax.set_visible(False)
    # No "N data sets" note in the panels: the one panel resting on fewer data sets
    # than the rest -- the log score -- is for the caption to state, and
    # `aggs[metric]["n_tasks"]` carries the count.
    fig.supxlabel(r"LorD rank $r$", fontsize=CFG.label_size)
    fig.supylabel(YLABEL, fontsize=CFG.label_size)
    handles, labels = _keys(axes)
    fig.legend(handles, labels, loc="outside upper center", ncol=len(labels),
               fontsize=CFG.legend_size, frameon=False)
    if save_as:
        pc.save(fig, save_as)
    return fig, axes, aggs


def _arm_only(agg, label):
    """One arm's rows plus the baseline: what a single-arm panel draws and scales."""
    return agg[(agg["kind"] == "baseline") | (agg["arm"] == label)]


def plot_arm_grid(df, metrics=METRICS, arms=ARMS, tasks=TASKS, balanced=True,
                  ncols=None, aspect=0.45, save_as=None, verbose=True,
                  how="median"):
    """The same pooled panels again, one row per arm, every panel on its own scale.

    The headline grid puts both arms in one panel, which is what makes the
    KS-against-NLL comparison readable -- and also what hides the KS arm: on a
    scale that has to reach 100x, a curve that stays within 2 % of ours is a
    line on 1 with invisible error bars. Giving each arm its own row and each
    panel its own limits gives that structure back, at the cost of the direct
    comparison. The two figures answer different questions, so both are kept.

    The admitted data sets and ranks are computed exactly as in
    :func:`plot_metric_grid` -- both arms are still required at a rank for it to
    enter -- so a row here is the same cell as the corresponding curve there,
    only drawn alone. Nothing but the axis limits changes between the two.

    `how` is `aggregate`'s: the default draws the median over data sets with
    the quartiles as the bar, `"arithmetic"` the mean with one standard
    deviation either side (the baseline's band then is the mean ± sd over its
    runs).
    """
    n_rows, n_cols = len(arms), len(metrics)
    fig, axes = pc.subplots(n_rows, n_cols, aspect=aspect, columns="double",
                            sharex=True, squeeze=False)
    aggs, ladders = {}, []
    for j, (metric, _label) in enumerate(metrics):
        if verbose:
            print(f"[{metric}]")
        axes[0][j].set_title(metric_label(metric), fontsize=CFG.title_size * 0.85)
        rat = ratios(df, metric, tasks, balanced, verbose)
        if rat.empty:
            for i in range(n_rows):
                axes[i][j].text(0.5, 0.5, "no runs yet", ha="center",
                                va="center", transform=axes[i][j].transAxes,
                                color=pc.COLORS["guide"],
                                fontsize=CFG.annotation_size * 0.8)
                axes[i][j].set_yticks([])
            continue
        scale = scale_of(metric)
        agg = aggregate(rat, how=how, scale=scale)
        aggs[metric] = agg
        ranks = _ladder(agg)
        ladders += ranks
        for i, arm in enumerate(arms):
            sub = _arm_only(agg, arm["label"])
            _draw(axes[i][j], sub, ranks)
            _yscale(axes[i][j], sub, scale)
    for ax in axes.flat:
        _decorate_x(ax, sorted(set(ladders)))
        # Six panels abreast of the text width leave about 1.1 in each, tick
        # labels included: at the paper's default size the rank ladder runs into
        # itself and "1.05" touches the panel to its left.
        ax.tick_params(labelsize=CFG.tick_size * 0.8)
    # No data-set note here either, as in `plot_metric_grid`.
    #for i, arm in enumerate(arms):
    #    axes[i][0].set_ylabel(arm["label"], color=arm["color"])
    # The rank belongs on the axis it parametrises, so it is labelled under the
    # panels and the key goes below that, in a box -- the same footer the other
    # ablation figures carry (`epoch_budget.ipynb`). Above the panels the key
    # would sit between the title of the section and the metric names, which is
    # where this figure's twelve panels need the room.
    _footer(fig, axes, r"LorD rank $r$")
    if save_as:
        pc.save(fig, save_as)
    return fig, axes, aggs

def plot_rank_ablation(df, metric=METRIC, tasks=TASKS, balanced=True, ax=None,
                       legend=True, save_as=None, verbose=True):
    """One metric on its own, single-column: the grid's panel, standalone."""
    fig = None
    if ax is None:
        fig, ax = pc.subplots(1, 1, aspect=0.72, columns="single")
    agg, ranks = _panel(ax, df, metric, tasks, balanced, verbose)
    if agg is None:
        raise ValueError(f"no task has a usable {metric} sweep")
    _decorate_x(ax, ranks)
    # Metric and sample size both go in the title. The y-label has to stay the
    # short shared one: at single-column width the panel is 2.3 in tall and a long
    # label would be clipped by the canvas. The sample size cannot go in a corner
    # either -- this panel carries a legend, and `loc="best"` hunts for the same
    # empty corner an annotation would.
    n = int(agg["n_tasks"].max())
    ax.set_title(f"{metric_label(metric)} "
                 f"({n} data set{'s' if n > 1 else ''})")
    ax.set_xlabel(r"LorD rank $r$")
    ax.set_ylabel(YLABEL)
    if legend:
        ax.legend(loc="best", fontsize=CFG.legend_size)
    if save_as and fig is not None:
        pc.save(fig, save_as)
    return fig, ax, agg


def plot_rank_ablation_by_task(df, metric=METRIC, tasks=TASKS, save_as=None):
    """Companion: the same ratio per dataset, so the pooled panel can be checked.

    Every dataset with a usable sweep appears, including the ones the balanced
    block drops -- this is where a half-finished campaign is supposed to be
    visible. Each panel is normalized by that dataset's own baseline and its
    error bar is that dataset's seed spread, so the y-axis means the same thing
    as in the pooled figure and the panels are directly comparable.
    """
    shown = [t for t in tasks
             if not ratios(df, metric, [t], balanced=False, verbose=False).empty]
    missing = [t["key"] for t in tasks if t not in shown]
    if missing:
        print(f"  no usable {metric} sweep on: {', '.join(missing)}")
    if not shown:
        raise ValueError(f"no task has a usable {metric} sweep")
    # Width follows the panel count instead of always claiming the text width:
    # a single-task companion should not be stretched across the page.
    width = min(pc.width_of("double"), 1.15 + 1.55 * len(shown))
    # 2.3 in of height, not the 1.9 the panels themselves need: the rotated
    # y-label is about 1.7 in of text and a shorter canvas clips it.
    fig, axes = pc.subplots(1, len(shown), aspect=2.3 / width, columns=width,
                            squeeze=False)
    for ax, t in zip(axes[0], shown):
        agg, ranks = _panel(ax, df, metric, [t], balanced=False, verbose=False)
        _decorate_x(ax, ranks)
        ax.set_title(t["label"])
        ax.set_xlabel(r"LorD rank $r$")
    # The panel titles are taken by the data set names, so the metric goes in
    # the y-label -- on its own line, because the two together are longer than
    # these panels are tall and a single-line label would be clipped. Not a
    # suptitle: `loc="outside upper center"` puts the legend in the same strip.
    axes[0][0].set_ylabel(f"{metric_label(metric)}\nagainst SCORE")
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncol=len(handles),
               fontsize=CFG.legend_size, frameon=False)
    if save_as:
        pc.save(fig, save_as)
    return fig, axes

In [ ]:
# The 2x3 grid, both arms in one panel per metric. Off by default: the per-arm
# figures below are the ones used (Figure 13). Set to True to draw it and write
# `ablation_rank_metrics.pdf`.
DRAW_METRIC_GRID = False
if DRAW_METRIC_GRID:
    fig, axes, aggs = plot_metric_grid(df, save_as="ablation_rank_metrics.pdf")
    display(aggs[METRIC])

The $2 \times 3$ grid (`plot_metric_grid`, off by default) is built for the
KS-against-NLL comparison and pays for it: the
NLL-fitted arm sets every panel's range — its upper quartile reaches
$1.56\times$ on MSE, $3.0\times$ on the kernel score, $5.4\times$ on CRPS and
$47\times$ on the variogram score, which puts four of the five ratio panels on a
log axis — and against it the
KS-fitted arm is a near-flat line on $1$. The figure below is the same data with one arm
per row and every panel on its own limits. The admitted data sets and ranks are
computed identically, so a row here is exactly the curve above drawn alone —
nothing but the axis limits differs.

That is enough to show the KS-fitted arm is *not* flat everywhere, and where it
is not, it is on the scores it was not fitted to. Read against SCORE's own seed
spread (the shaded strip on $1$: $\pm 0.22\,\%$ on the energy score and on CRPS,
$\pm 0.32$–$0.34\,\%$ on the kernel score, the variogram score and MSE), its
median over the eleven data sets is worse by $2.7$–$3.6\,\%$ on CRPS,
$4.7$–$5.2\,\%$ on the variogram score, $0.4$–$0.8\,\%$ on MSE and
$0.2$–$0.5\,\%$ on the energy score, and **level with SCORE on the kernel score**
($0.998$–$1.000$, inside SCORE's own band at every rank). It is clear of the band on
CRPS by $12.1$–$16.2$ seed spreads, on the variogram score by $13.8$–$15.5$, on
MSE by $1.2$–$2.3$ and on the energy score by only $0.9$–$2.2$. The box reaches
below $1$ on the kernel and the energy score at every rank, on MSE at $r = 8$
and $16$ and on the variogram score at $r = 16$, and never on CRPS: depth beats
SCORE by $3$–$7\,\%$ on MSE, CRPS, the energy and the kernel score and by
$0.5$–$1.8\,\%$ on the variogram score, EUPPBench by up to $1.4\,\%$ on MSE,
the energy and the kernel score, traffic by $0.2\,\%$ on the energy score and
$1\,\%$ on the kernel score, and on the kernel score illness, ETTm2 and weather
join them, so six of the eleven data sets sit at or under the SCORE line there.
Illness is the *worst* data set for the KS arm on MSE ($1.06$–$1.14\times$,
traffic next at $1.05\times$), traffic on CRPS ($1.32$–$1.35\times$, then weather
at $1.09$–$1.12\times$, electricity at $1.08$–$1.11\times$ and illness at
$1.05$–$1.11\times$), and electricity and ERA5 on the variogram score
($1.13$–$1.18\times$ and $1.09$–$1.18\times$, traffic next at $1.12$–$1.13\times$
and illness at $1.05$–$1.13\times$). Traffic is the data set where SCORE's
lead on the marginals is widest and its lead on the joint scores narrowest.

**Rank narrows the gap on CRPS and closes it nowhere.**
On CRPS the KS arm's median goes from $1.036$ at $r = 1$ to $1.027$ at $r = 16$
— ERA5 ($1.07$ to $1.04$), electricity ($1.11$ to $1.08$), weather ($1.12$ to
$1.10$), traffic ($1.35$ to $1.32$), ETTh1 ($1.04$ to $1.03$) and ETTm1 ($1.03$
to $1.02$) narrow, illness widens ($1.05$ to $1.08$) — while the energy score
barely moves ($1.002$–$1.005$, $1.004$ at $r = 1$ and $1.003$ at $r = 16$). The
variogram score is $1.052$ at $r = 1$ and $1.047$–$1.048$ at every higher rank:
ERA5 ($1.18$ to $1.09$) and ETTm1 ($1.03$ to $0.99$, its univariate horizons
falling to $0.97$–$0.98$) narrow the whole way, but a median of eleven is the
sixth value, and from $r = 2$ on that is ETTh1, flat at $1.05$; electricity, the
worst data set from $r = 4$ on, widens ($1.13$ to $1.18$), and traffic stays at
$1.12$–$1.13$. MSE ($1.004$–$1.008$) and the kernel score ($0.998$–$1.000$) stay
level. No data set's curve has to be monotone for
the median to move like this, and several are not —
`plot_rank_ablation_by_task(df, metric)` draws the per-task view that shows it.

**The log-score panel is the one place the two rows disagree,** and it is the
panel the difference scale bought. The NLL-fitted arm is the only arm anywhere
in this figure that *converges* on SCORE: $+0.56$ nats per coordinate at $r = 1$,
$+0.17$ at $r = 4$ and $r = 8$ and $-0.06$ at $r = 16$ — given enough rank, a
core fitted to the log score approaches SCORE's density, which is what the
expressivity claim predicts, and at $r = 16$ it **passes** it on five of the nine
data sets (ETTh2, ETTm2, EUPPBench, traffic and weather), which puts the median
just below the SCORE line. Weather is one of the five only because its reference is
degenerate (above) — the NLL arm sits $335$–$338$ nats below it at every rank —
and without it the median at $r = 16$ is $+0.06$. The genuine passes are traffic,
where the NLL arm is $1.04$–$1.10$ nats below SCORE at *every* rank (SCORE's
log score trails every Gaussian fit's there, $+1.3$–$1.4$ nats per coordinate
against $+0.2$–$0.4$ for the NLL fits), and ETTm2 ($-0.91$ nats at $r = 16$, from its
univariate pair). The KS-fitted arm does not converge: $+0.60$ nats at $r = 1$
and $+1.38$ at $r = 16$ (dipping to $+0.55$ at $r = 2$), while its upper
quartile widens from $+1.04$ to $+2.59$ — most data sets become *less*
density-calibrated as capacity grows (ETTm1 from $+1.04$ to $+2.75$, illness from
$+0.41$ to $+2.59$, ETTh2 from $+0.60$ to $+1.46$, and weather, from $67$ nats
below SCORE at $r = 1$ to $119$ above), only ERA5 ($+1.82$ to $+1.38$) and ETTm2
($+1.56$ to $+0.44$) improve, and traffic sits flat below SCORE ($-0.55$ to
$-0.58$). The two start $0.05$ nats apart at $r = 1$ and are $1.44$ apart at
$r = 16$ ($0.05$ and $1.12$ without weather), so the objective that wins on every
proper score is also the one whose forecast cannot be read as a density — and
the figure shows both halves of that on nine data sets.

So on the proper scores the KS-fitted low-rank core is *nearly* indistinguishable
from SCORE — level on the kernel score it was fitted to, within one percent on
MSE and the energy score, three to four on CRPS, and $4.7$–$5.2$ percent behind
on the variogram score — against the multiple the NLL row shows ($1.10\times$ to
$1.86\times$ on the medians, rising with rank on every one of the five). On the log score the ordering of the two arms
is reversed.

In [ ]:
fig, axes, aggs = plot_arm_grid(df, save_as="ablation_rank_metrics_by_arm.pdf", aspect = 0.28)

### Mean and standard deviation instead of the quartiles

The per-arm figure once more with `how="arithmetic"`:
a point is the **mean** over the eleven data sets (nine on the log score) and its
bar **one standard deviation** either side, in place of the median and the
$0.25$–$0.75$ quartiles. Nothing before that last step changes -- seeds are
averaged within a task and a data set's tasks collapsed by their median, as
above -- and the SCORE band is the mean $\pm$ sd of the baseline's individual runs.
Written to `ablation_rank_metrics_by_arm_mean_sd.pdf`.

What the swap shows is how far a few data sets carry a mean:

* **The KS arm barely moves; its bars widen.** The means sit within a few per
  cent of SCORE, as the medians do -- MSE $1.009$–$1.017\times$, CRPS
  $1.055$–$1.064\times$, ES $1.000$–$1.003\times$, VS $1.053$–$1.060\times$, KS
  $0.997$–$0.999\times$ -- but the bars are now the data sets' full spread:
  $\pm 1.5$–$1.8\,\%$ on the kernel score, $\pm 1.9$–$2.4\,\%$ on ES,
  $\pm 2.9$–$4.6\,\%$ on MSE, $\pm 6\,\%$ on VS and $\pm 10$–$11\,\%$ on CRPS,
  where traffic's $1.32$–$1.35\times$ doubles the spread and lifts the mean
  $3\,\%$ above the median. The SCORE band widens as well, to $\pm 0.7$–$1.2\,\%$
  (against $\pm 0.2$–$0.35\,\%$ as quartiles).
* **The NLL arm's means are depth's.** $3.6$–$4.5\times$ on MSE, $2.5$–$5.6\times$
  on CRPS, $2.2$–$4.7\times$ on ES, $7.8$–$21\times$ on KS and $9$–$108\times$ on
  VS, against medians of $1.10$–$1.86\times$: depth ($26$–$37\times$ on MSE, up to
  $890\times$ on VS), with ERA5, electricity and traffic behind it. The sd exceeds the
  mean at every rank, so every lower bar reaches below zero; on the log panels
  it is cut off at the axis.
* **The log score is weather's.** The NLL arm sits at $-37$ to $-38$ nats
  $\pm 112$ at every rank -- weather's $-337$ shared among nine data sets -- and
  the KS arm goes from $-6.8$ at $r = 1$ to $+14.2$ at $r = 16$ ($\pm 5$–$39$);
  the SCORE band is $\pm 7.3$ nats, weather's seed spread. The NLL arm's convergence
  on SCORE, which the median figure shows, is not visible on this scale.

So the mean figure shows magnitudes and the median figure what a typical data
set does; the prose of this section quotes the median figure.

In [ ]:
# The per-arm figure with the mean and one standard deviation over data sets in
# place of the median and the quartiles.
fig, axes, aggs_mean = plot_arm_grid(
    df, how="arithmetic", aspect=0.28, verbose=False,
    save_as="ablation_rank_metrics_by_arm_mean_sd.pdf")

### Presentation copy — per-arm grid, four metrics

The per-arm figure above rebuilt with the same function and the same admitted data sets, restricted
to **MSE, CRPS, ES and VS** (the kernel-score and log-score columns dropped), and with taller
panels: `aspect=0.5` against the paper's `0.28`, so the four columns come out closer to square
than the six-column strip. Written to `figures/presentation/` (git-ignored), never over the paper
figure.

In [ ]:
#: The four metrics the presentation copies keep, in the grid's own order.
PRESENTATION_METRICS = [m for m in METRICS
                        if m[0] in ("mse", "crps", "energy_score", "variogram_score")]

fig, axes, _ = plot_arm_grid(df, metrics=PRESENTATION_METRICS, aspect=0.5, verbose=False,
                             save_as=pc.PRESENTATION_DIR / "ablation_rank_metrics_by_arm.pdf")
plt.show()
